# 10. Создание нейросетей

Видео: [10. Создание нейросетей](https://www.youtube.com/watch?v=YiatESITMAg&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=11)

## О чём урок

Видео 9 уже закрыло **сборку классов** (`Sequential`, `nn.Module`, `ModuleList`/`Dict`, `add_module`).  
У автора в видео 10 часть этого **повторяется** — в тренажёре дубль не делаем.

Здесь только то, что в 09 не тренировали:

| блок | тема |
|:---|:---|
| 1 | `state_dict` — снимок весов |
| 2 | `parameters()` → Adam, число параметров |
| 3 | `train()` / `eval()` |
| 4 | CE / MSE + Adam на двух готовых сетях (как в конце видео) |

> **Конспект:** голова под задачу — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§10**; чеклист — **§12**. Классы — [`09/`](../09/09_классы_нейросетей.ipynb). Карта — [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md).

**Как смотреть.** ~11 мин. Повтор классов у автора можно **пролистать**; пауза на `state_dict` / `parameters` / `train`·`eval` и на финальную склейку с Adam.

**Как решать.** Замени `...` → проверка. Решения — `10_создание_нейросетей_решения.ipynb` после попытки.

In [8]:
import torch
import torch.nn as nn

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"


print("torch", torch.__version__)

torch 2.14.1+cpu


## 1. <span style="color:#7CDBA0">state_dict</span> — снимок весов

> **Конспект:** веса слоёв — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§7**, **§12**.

`model.state_dict()` — **словарь** ключ → тензор параметра.

В `Sequential` без имён ключи — индексы: `0.weight`, `0.bias`, …  
**ReLU не попадает** в словарь: нет обучаемых весов.

Форма `Linear(784, 128).weight` — `(128, 784)`. Позже (видео 13) этот словарь сохраняют на диск.

In [9]:
# 1.1 state_dict: ключи и форма весов
torch.manual_seed(0)
net = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

sd = net.state_dict()
keys = list(sd.keys())
w0 = sd["0.weight"]  # ReLU = слой 1, весов нет → ключи 0 и 2
print("keys:", keys)
print("w0 shape:", tuple(w0.shape))

keys: ['0.weight', '0.bias', '2.weight', '2.bias']
w0 shape: (128, 784)


In [10]:
need(isinstance(net, nn.Sequential), "net = nn.Sequential(...)")
need(isinstance(sd, dict), "sd = net.state_dict()")
need(keys == ["0.weight", "0.bias", "2.weight", "2.bias"], f"keys={keys} — у ReLU нет весов, индексы 0 и 2")
need_tensor(w0, "w0")
need(tuple(w0.shape) == (128, 784), f"w0.shape={tuple(w0.shape)}, ждали (128, 784)")
print("1.1 — верно")

1.1 — верно


## 2. <span style="color:#C9A0DC">parameters()</span> → оптимизатор

`model.parameters()` — все обучаемые тензоры. Их отдают в Adam:

```python
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
```

Число параметров: `sum(p.numel() for p in model.parameters())`.  
Для `784→128→10`: `784·128 + 128 + 128·10 + 10 = 101770`.

In [11]:
# 2.1 parameters() → Adam + число параметров
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

# Adam(model.parameters(), lr=1e-3)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)               
# sum(p.numel() for p in model.parameters())
n_params = sum(p.numel() for p in model.parameters())          
# 784*128 + 128 + 128*10 + 10
n_manual = 784 * 128 + 128 + 128 * 10 + 10         

print("n_params:", n_params, "| manual:", n_manual, "| opt:", type(opt).__name__)

n_params: 101770 | manual: 101770 | opt: Adam


In [12]:
need(isinstance(opt, torch.optim.Adam), "opt = torch.optim.Adam(model.parameters(), lr=1e-3)")
need(n_params == 101770, f"n_params={n_params}, ждали 101770")
need(n_manual == n_params, f"n_manual={n_manual} должно совпасть с n_params")
print("2.1 — верно")

2.1 — верно


## 3. <span style="color:#E6C07B">train()</span> и <span style="color:#E6C07B">eval()</span>

| вызов | `model.training` | когда |
|:---|:---|:---|
| `model.train()` | `True` | перед обучением |
| `model.eval()` | `False` | перед val / test |

Сейчас без Dropout/BatchNorm разницы почти нет, но **привычка нужна** с видео 15–16. Где ставить в цикле — видео 11.

In [15]:
# 3.1 train() / eval() — флаг model.training
m = nn.Linear(4, 2)
was_train = m.training
m.eval()
after_eval = m.training
m.train()
after_train = m.training

print("start:", was_train, "| after eval:", after_eval, "| after train:", after_train)

start: True | after eval: False | after train: True


In [16]:
need(was_train is True, "новая модель уже в train-режиме (training=True)")
need(after_eval is False, "после eval() → training=False")
need(after_train is True, "после train() → training=True")
print("3.1 — верно")

3.1 — верно


## 4. Loss + Adam к двум сетям (конец видео 10)

Класс модели **не пишем заново** — это уже `09/` и проект (видео 9).  
Здесь как у автора в финале: две сети + свой loss + свой Adam.

| задача | вход (уже вектор) | выход | loss | opt |
|:---|:---|:---|:---|:---|
| классификация | `(B, 784)` | 10 | `CrossEntropyLoss` | `Adam(clf.parameters())` |
| регрессия | `(B, 4096)` | 2 | `MSELoss` | `Adam(reg.parameters())` |

Сети ниже уже созданы — допиши только loss и оптимизаторы, проверь `shape`.

In [17]:
# 4.1 CE/MSE + Adam (сети готовы — класс не трогаем)
torch.manual_seed(0)
clf = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
reg = nn.Sequential(nn.Linear(64 * 64, 128), nn.ReLU(), nn.Linear(128, 2))

criterion_clf = nn.CrossEntropyLoss()
opt_clf = torch.optim.Adam(clf.parameters(), lr=1e-3)
criterion_reg = nn.MSELoss()
opt_reg = torch.optim.Adam(reg.parameters(), lr=1e-3)

y_clf = clf(torch.randn(8, 784))
y_reg = reg(torch.randn(8, 64 * 64))
print(
    "clf:", tuple(y_clf.shape) if isinstance(y_clf, torch.Tensor) else y_clf,
    "| reg:", tuple(y_reg.shape) if isinstance(y_reg, torch.Tensor) else y_reg,
)
print("losses:", type(criterion_clf).__name__, type(criterion_reg).__name__)

clf: (8, 10) | reg: (8, 2)
losses: CrossEntropyLoss MSELoss


In [18]:
need(isinstance(criterion_clf, nn.CrossEntropyLoss), "criterion_clf = nn.CrossEntropyLoss()")
need(isinstance(criterion_reg, nn.MSELoss), "criterion_reg = nn.MSELoss()")
need(isinstance(opt_clf, torch.optim.Adam) and isinstance(opt_reg, torch.optim.Adam), "Adam(...parameters()...)")
need_tensor(y_clf, "y_clf")
need_tensor(y_reg, "y_reg")
need(tuple(y_clf.shape) == (8, 10), f"clf out {tuple(y_clf.shape)}")
need(tuple(y_reg.shape) == (8, 2), f"reg out {tuple(y_reg.shape)}")
print("4.1 — верно")

4.1 — верно


## 5. Что мы сделали

| тема | зачем |
|:---|:---|
| `state_dict` | смотреть / потом сохранять веса |
| `parameters` + Adam | что обновляет градиентный спуск |
| `train` / `eval` | режимы до Dropout/BN и для цикла |
| CE/MSE + Adam | склейка к двум задачам без повторного класса |

Дубли из видео 9 (`Sequential` с нуля, свой `Module`, `add_module`, ModuleList) — сознательно убраны.

Дальше: раздел **«Видео 10»** в `project/fashion_mnist.ipynb`. Цикл эпох — видео 11.